In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')
import os, json, time
import numpy as np, pandas as pd
import torch
from transformers import AutoModel

BASE = '/content/drive/MyDrive/standup4ai'
AUDIO_DIR = BASE + '/audio_1000'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'
OUT_DIR = BASE + '/wordlevel_wavlm_features'
CKPT_FILE = BASE + '/wordlevel_wavlm_checkpoint.json'
os.makedirs(OUT_DIR, exist_ok=True)

# Load checkpoint
done = set()
if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE) as f:
        done = set(json.load(f).get('done', []))

# Build audio lookup (handles both 'vid.m4a' and 'vid,lang.m4a')
audio_map = {}
if os.path.exists(AUDIO_DIR):
    for f in os.listdir(AUDIO_DIR):
        if not f.endswith(('.m4a', '.wav', '.mp3', '.webm')):
            continue
        base = f.rsplit('.', 1)[0]
        if ',' in base:
            base = base.split(',')[0]  # strip lang suffix
        audio_map[base] = os.path.join(AUDIO_DIR, f)

# Build label lookup
label_map = {}
if os.path.exists(LABEL_DIR):
    for f in os.listdir(LABEL_DIR):
        if f.endswith('.csv'):
            label_map[f.replace('.csv', '')] = os.path.join(LABEL_DIR, f)

# Videos with BOTH audio AND labels, not yet done
overlap = sorted(set(audio_map.keys()) & set(label_map.keys()) - done)
print(f'Audio: {len(audio_map)} | Labels: {len(label_map)} | To process: {len(overlap)}')
print(f'Already done: {len(done)}')


In [ ]:
# Load WavLM on GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('Loading WavLM-base...')
wavlm = AutoModel.from_pretrained('microsoft/wavlm-base')
wavlm.to(device)
wavlm.eval()
print('WavLM ready!')


In [ ]:
# Per-word feature extraction — simplest possible approach
SR = 16000
MIN_DUR = 0.02  # 20ms minimum

def parse_timestamp(ts):
    ts = str(ts).strip().strip('[]')
    p = ts.split(',')
    return float(p[0]), float(p[1])

def get_word_audio(audio_path, t0, t1):
    dur = t1 - t0
    if dur <= 0:
        return np.zeros(320, dtype=np.float32)  # 20ms at 16kHz
    # Use librosa with offset/duration — load FULL audio then slice
    try:
        import librosa
        y, _ = librosa.load(audio_path, offset=t0, duration=dur, sr=SR, mono=True)
        if len(y) < int(MIN_DUR * SR):
            y = np.pad(y, (0, int(MIN_DUR * SR) - len(y)))
        return y.astype(np.float32)
    except Exception as e:
        return np.zeros(int(MIN_DUR * SR), dtype=np.float32)

def extract_features(audio_path, word_times, batch_size=32):
    n = len(word_times)
    if n == 0:
        return None
    feats = []
    for i in range(0, n, batch_size):
        batch_times = word_times[i:i+batch_size]
        segments = [get_word_audio(audio_path, t0, t1) for t0, t1 in batch_times]
        max_len = max(len(s) for s in segments)
        padded = []
        for s in segments:
            if len(s) < max_len:
                s = np.pad(s, (0, max_len - len(s)))
            padded.append(s)
        batch = torch.tensor(np.stack(padded), dtype=torch.float32).to(device)
        with torch.no_grad():
            out = wavlm(batch).last_hidden_state  # (batch, seq, 768)
            emb = out.mean(dim=1).squeeze(1)   # (batch, 768)
        feats.append(emb.cpu().numpy())
    return np.vstack(feats)

print('Extractor ready')


In [ ]:
# Process all videos
SKIP_MIN_WORDS = 30  # Skip videos with fewer words
t0 = time.time()
for i, vid in enumerate(overlap):
    out_file = OUT_DIR + '/' + vid + '_word_features.npy'
    if os.path.exists(out_file):
        continue

    audio_path = audio_map[vid]
    df = pd.read_csv(label_map[vid])

    word_times, word_labels = [], []
    for _, row in df.iterrows():
        try:
            t0_w, t1_w = parse_timestamp(row['timestamp'])
            word_times.append((t0_w, t1_w))
            word_labels.append(str(row['label']).strip())
        except:
            continue

    if len(word_times) < SKIP_MIN_WORDS:
        done.add(vid)
        continue

    feats = extract_features(audio_path, word_times)
    if feats is None or len(feats) == 0:
        continue

    assert len(feats) == len(word_labels), f'{vid}: {len(feats)} != {len(word_labels)}'
    np.save(out_file, feats)
    done.add(vid)

    elapsed = time.time() - t0
    rate = (i + 1) / elapsed * 3600 if elapsed > 0 else 0
    print(f'{i+1}/{len(overlap)} {vid}: {feats.shape} | done={len(done)} | {rate:.0f}/hr')

    if len(done) % 10 == 0:
        with open(CKPT_FILE, 'w') as f:
            json.dump({'done': list(done)}, f)

with open(CKPT_FILE, 'w') as f:
    json.dump({'done': list(done)}, f)
print(f'\nDone: {len(done)}/{len(overlap)} videos in {(time.time()-t0)/60:.0f} min')


In [ ]:
# Summary
feat_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith('_word_features.npy')])
print(f'Word-level features: {len(feat_files)} videos')
for f in feat_files[:5]:
    d = np.load(OUT_DIR + '/' + f)
    print(f'  {f}: {d.shape}')
